# 🛡️ CryptoGraph-Secure: Market Dynamics & Diebold-Yilmaz Spillover Baseline

This research notebook analyzes the **10-year dynamic cryptocurrency network** and establishes the **econometric statistical baseline** required by our research rubric:

1. **Exploratory Data Analysis (EDA):** Inspect 10-year daily historical prices, return distributions, and volatility across the top 20 cryptocurrencies (`2016–2026`).
2. **Dynamic Graph Evolution:** Trace how inter-asset network density and edge counts spike during systemic market panics.
3. **Statistical Baseline (Diebold-Yilmaz 2012):** Fit a Vector Autoregression (VAR) with Generalized Forecast Error Variance Decomposition (GFEVD) to construct the **Directional Spillover Table**.
4. **NET Connectedness Ranking:** Quantify which assets are net **shock transmitters** vs. net **shock receivers** to serve as the statistical ground-truth for our **TempME** temporal motif explainer validation.

In [2]:
import os
import sys
from pathlib import Path

# Add src directory to path
src_path = str(Path("../src").resolve())
if src_path not in sys.path:
    sys.path.insert(0, src_path)

import torch
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from cryptograph.data.dataloader import load_processed_dataset
from cryptograph.models.baselines.var_spillover import VARSpilloverBaseline
from cryptograph.spillover.shock_detector import ShockDetector

print("All modules imported successfully!")

c:\Users\daksh\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


All modules imported successfully!


## 1. Load Processed 10-Year Dynamic Graph Dataset

We load the serialized PyTorch tensors created by our feature builder and dynamic graph pipeline (`data/processed/crypto_graph_dataset.pt`).

In [3]:
dataset_path = Path("../data/processed/crypto_graph_dataset.pt")
data = load_processed_dataset(dataset_path)

features = data["features"]        # (T, num_nodes, num_features)
targets = data["targets"]          # (T, num_nodes, horizons)
active_mask = data["active_mask"]  # (T, num_nodes)
edge_indices = data["edge_indices"]
edge_weights = data["edge_weights"]
asset_names = data["asset_names"]
timestamps = pd.to_datetime(data["timestamps"])
feature_names = data["feature_names"]
horizons = data["horizons"]

print(f"✅ Loaded {features.shape[0]} daily timesteps for {len(asset_names)} assets.")
print(f"Features per node ({len(feature_names)}): {feature_names}")
print(f"Forecast horizons (days): {horizons}")
print(f"Date Range: {timestamps[0].strftime('%Y-%m-%d')} to {timestamps[-1].strftime('%Y-%m-%d')}")

✅ Loaded 3653 daily timesteps for 20 assets.
Features per node (9): ['log_return', 'volatility_7', 'volatility_14', 'volatility_30', 'hl_spread', 'volume_ratio', 'log_volume', 'rsi_14', 'macd']
Forecast horizons (days): [1, 3, 7, 14, 30]
Date Range: 2016-09-24 to 2026-09-24


## 2. 10-Year Historical Price Trajectories & Crisis Timeline

Let's reconstruct the 10-year cumulative prices from log returns and annotate iconic market shocks:
- **March 2020:** COVID-19 Global Liquidity Crash
- **November 2021:** Crypto Bull Run Peak (All-Time High)
- **May 2022:** Terra/Luna Algorithmic Depeg Collapse
- **November 2022:** FTX Exchange Bankruptcy & Contagion
- **March 2024:** 2024 Market High

In [4]:
# Extract log returns (feature index 0) and reconstruct indexed cumulative returns
log_returns_df = pd.DataFrame(
    features[:, :, 0].numpy(),
    index=timestamps,
    columns=asset_names
)

# Select top benchmark assets for clean visualization
selected_assets = ["BTC", "ETH", "SOL", "BNB", "DOGE", "ADA", "LINK", "AVAX"]
cum_returns = np.exp(log_returns_df[selected_assets].cumsum())

fig_price = go.Figure()
for asset in selected_assets:
    fig_price.add_trace(go.Scatter(
        x=cum_returns.index,
        y=cum_returns[asset],
        mode="lines",
        name=asset
    ))

# Add Crisis Annotations
crises = {
    "2020-03-12": "COVID Crash",
    "2021-11-10": "2021 ATH Peak",
    "2022-05-09": "Terra/Luna Depeg",
    "2022-11-09": "FTX Collapse",
    "2024-03-13": "2024 ATH Run"
}

for date_str, label in crises.items():
    fig_price.add_vline(
        x=date_str,
        line_width=1.5,
        line_dash="dash",
        line_color="red"
    )
    fig_price.add_annotation(
        x=date_str,
        y=1,
        yref="paper",
        text=label,
        showarrow=False,
        textangle=-90,
        xanchor="right",
        font=dict(color="red", size=10)
    )

fig_price.update_layout(
    title="10-Year Cumulative Growth & Major Systemic Shocks (Log Scale)",
    yaxis_type="log",
    xaxis_title="Date",
    yaxis_title="Growth Factor (Log Scale)",
    template="plotly_dark",
    height=700
)
fig_price.show()

## 3. Dynamic Graph Evolution: Panic Contagion Effect

In dynamic graph neural networks, the number of active edges changes over time. During quiet regimes, only closely related coins correlate. During systemic panics, market-wide sell-offs cause correlations to spike, creating dense graph clusters.

In [5]:
# Extract number of edges across all 3,653 snapshots
edge_counts = [e.size(1) for e in edge_indices]
edge_series = pd.Series(edge_counts, index=timestamps)

fig_edges = go.Figure()
fig_edges.add_trace(go.Scatter(
    x=edge_series.index,
    y=edge_series,
    mode="lines",
    name="Active Edges",
    line=dict(color="#00e676", width=1.5)
))

# Overlay 30-day moving average of graph density
fig_edges.add_trace(go.Scatter(
    x=edge_series.index,
    y=edge_series.rolling(30).mean(),
    mode="lines",
    name="30-day Moving Avg",
    line=dict(color="#ffea00", width=2)
))

fig_edges.update_layout(
    title="Dynamic Graph Edge Count (Network Density Over Time)",
    xaxis_title="Date",
    yaxis_title="Total Dynamic Edges",
    template="plotly_dark",
    height=450
)
fig_edges.show()

## 4. Cross-Asset Correlation Heatmap

Let's visualize the pairwise correlation matrix across all 20 cryptocurrencies.

In [6]:
all_active_mask = active_mask.numpy().all(axis=1)
corr_matrix = log_returns_df.loc[all_active_mask].corr()

fig_corr = px.imshow(
    corr_matrix,
    x=asset_names,
    y=asset_names,
    color_continuous_scale="Viridis",
    title="Pairwise Correlation Matrix (Top 20 Cryptocurrencies)",
    labels=dict(color="Correlation")
)
fig_corr.update_layout(template="plotly_dark", width=700, height=650)
fig_corr.show()

## 5. Statistical Baseline: Diebold-Yilmaz Spillover Index (VAR)

Now we fit the **VAR-based Diebold-Yilmaz (2012) Spillover Model** using Generalized Forecast Error Variance Decomposition (GFEVD).

This computes:
- **Spillover TO others:** Total variance transmitted from coin $i$ to all other coins.
- **Spillover FROM others:** Total variance received by coin $i$ from all other coins.
- **NET Spillover:** $\text{NET}_i = \text{TO}_i - \text{FROM}_i$. A positive value indicates a **net shock transmitter**, while a negative value indicates a **net shock receiver**.
- **Total Spillover Index (TSI):** System-wide percentage of forecast error variance coming from spillovers.

In [7]:
# Fit VAR Spillover Baseline on the common period where all 20 assets are active (2,131 days)
common_returns = log_returns_df.loc[all_active_mask]
baseline = VARSpilloverBaseline(maxlags=2, forecast_horizon=10)
baseline.fit(common_returns)

spillover_results = baseline.compute_spillover_table()
tsi = spillover_results["total_spillover_index"]
net_spillover = spillover_results["net_spillover"].sort_values(ascending=False)

print(f"═══════════════════════════════════════════════════════════")
print(f"📊 Total Spillover Index (TSI): {tsi:.2f}%")
print(f"═══════════════════════════════════════════════════════════")
print("\n🏆 Top Systemic Shock Transmitters (NET > 0):")
print(net_spillover[net_spillover > 0])
print("\n🛡️ Top Systemic Shock Receivers (NET < 0):")
print(net_spillover[net_spillover < 0])

[2026-09-27 12:58:09] [INFO] [cryptograph.models.baselines.var_spillover] - Fitted VAR baseline with lag order: 2
═══════════════════════════════════════════════════════════
📊 Total Spillover Index (TSI): 54.24%
═══════════════════════════════════════════════════════════

🏆 Top Systemic Shock Transmitters (NET > 0):
BTC    731.495421
ETH     52.795962
dtype: float64

🛡️ Top Systemic Shock Receivers (NET < 0):
XRP     -3.919257
DOGE   -18.188727
BNB    -20.517609
SHIB   -20.909920
SOL    -30.681656
TRX    -33.121533
XMR    -39.683279
ADA    -41.119825
BCH    -45.061506
AVAX   -49.695169
NEAR   -50.207858
FIL    -53.182575
LINK   -55.098574
DOT    -57.435271
ATOM   -62.764210
ETC    -66.863311
LTC    -67.243521
XLM    -68.597582
dtype: float64


c:\Users\daksh\AppData\Local\Programs\Python\Python311\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)


## 6. NET Systemic Connectedness Ranking

Let's visualize the **NET Connectedness Ranking**. This ranking is exported to disk and will serve as the scientific benchmark when validating our **TempME Temporal Motif Explainer** via Spearman rank correlation.

In [8]:
net_df = pd.DataFrame({
    "Asset": net_spillover.index,
    "Net_Spillover": net_spillover.values,
    "Type": ["Net Transmitter" if v > 0 else "Net Receiver" for v in net_spillover.values]
})

fig_net = px.bar(
    net_df,
    x="Asset",
    y="Net_Spillover",
    color="Type",
    color_discrete_map={"Net Transmitter": "#00e676", "Net Receiver": "#ff1744"},
    title=f"Diebold-Yilmaz NET Systemic Spillover Ranking (TSI = {tsi:.2f}%)",
    labels={"Net_Spillover": "NET Spillover (TO - FROM) %"}
)
fig_net.update_layout(template="plotly_dark", height=450)
fig_net.show()

# Export statistical ranking for TempME validation
output_ranking_path = Path("../data/processed/statistical_connectedness.csv")
net_df.to_csv(output_ranking_path, index=False)
print(f"✅ Saved statistical connectedness ranking to: {output_ranking_path}")

✅ Saved statistical connectedness ranking to: ..\data\processed\statistical_connectedness.csv


## 7. Key Takeaways & Transition to GNN Modeling

- **Connectedness Findings:** Bitcoin (BTC, +731.5) and Ethereum (ETH, +52.8) emerge as the primary systemic **transmitters**, while high-beta altcoins absorb volatility as **receivers**.
- **Next Step (Notebook 02):** Train the **CryptoTGN** model, extract dynamic shock transmission motifs using **TempME**, and evaluate whether the GNN's temporal explanations align with and refine this statistical connectedness ranking using **Spearman Rank Correlation** and **Fidelity+**.